In [2]:
import pandas as pd
df = pd.read_csv('/Users/annu/Desktop/Financial Misinformation Detection with LLM Hallucination Auditing + Market Impact Analysis/data/headlines_verified.csv')
def build_prompt(headline):
    return f"""You are a financial fact-checking assistant specializing in Indian and global financial markets.

Your task is to classify the following financial news headline as either FACTUAL or MISLEADING.

- FACTUAL: The headline accurately represents a real financial event, uses precise language, and does not exaggerate or distort facts.
- MISLEADING: The headline exaggerates, distorts, omits critical context, uses sensational language, or is designed to manipulate market sentiment.

Headline: "{headline}"

Respond in this exact format and nothing else:
CLASSIFICATION: <FACTUAL or MISLEADING>
CONFIDENCE: <HIGH or MEDIUM or LOW>
REASONING: <one sentence explaining your decision>"""

In [ ]:

from groq import Groq
import time
from dotenv import load_dotenv
import os
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_content(prompt):
    chat = client.chat.completions.create(
      model="llama-3.1-8b-instant",  # smaller = faster response time
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return chat.choices[0].message.content


#(The Request)
response = []

def parse_response(text):
    lines = text.strip().split("\n")
    lines = [l for l in lines if l.strip()]
    classification = lines[0].split(':')[1].strip()
    confidence = lines[1].split(':')[1].strip()
    reasoning = lines[2].split(':')[1].strip()

    return {
        'classification' : classification,
        'confidence': confidence,
        'reasoning' : reasoning
    }

for index, row in df.iterrows():
    single_headline = row['headline']
    prompt = build_prompt(single_headline)
    row_response = generate_content(prompt)
    parsed = parse_response(row_response)
    response.append(parsed)
    time.sleep(1)

print(response)






[{'classification': 'MISLEADING', 'confidence': 'HIGH', 'reasoning': 'Although the Hindenburg Research report did accuse Adani Group of serious accounting allegations, the headline phrase "brazen stock manipulation and accounting fraud scheme over decades" is overly dramatic and sensational, exaggerating the scope and severity of the allegations.'}, {'classification': 'FACTUAL', 'confidence': 'HIGH', 'reasoning': "The headline accurately reports on the Adani Group's response to the Hindenburg report, using specific details such as the number of pages in the rebuttal, which suggests a high level of factual accuracy."}, {'classification': 'MISLEADING', 'confidence': 'MEDIUM', 'reasoning': 'The headline omits critical context by implying a sudden three-day collapse without specifying the starting point of the loss or the specific events leading to it.'}, {'classification': 'FACTUAL', 'confidence': 'HIGH', 'reasoning': 'The headline accurately reports on a specific incident related to Adan

In [4]:
import pandas as pd

df = pd.read_csv('headlines_layer1.csv')
print(df.columns.tolist())

['date', 'headline', 'source', 'related_stock', 'category', 'news_type', 'classification', 'confidence', 'reasoning']


In [5]:
results_df = pd.DataFrame(response)
final_df = pd.concat([df.reset_index(drop=True), results_df], axis=1)
final_df.to_csv('headlines_layer1.csv', index=False)
print(final_df.shape)

(87, 12)


In [11]:
df = pd.read_csv('headlines_layer1.csv')
# keep only the first occurrence of each column
df = df.loc[:, ~df.columns.duplicated()]
df.to_csv('headlines_layer1.csv', index=False)
print(df.columns.tolist())

['date', 'headline', 'source', 'related_stock', 'category', 'news_type', 'classification', 'confidence', 'reasoning']


In [ ]:
# only keep these exact columns
df_final = df[['date', 'headline', 'source', 'related_stock', 'category', 'news_type', 'classification', 'confidence', 'reasoning']]
df_final.to_csv('headlines_layer1.csv', index=False)
print(df_final.columns.tolist())

Saved cleanly
['date', 'headline', 'source', 'related_stock', 'category', 'news_type', 'classification', 'confidence', 'reasoning']
